# Question Answering with LangChain, OpenAI, and MultiQuery Retriever

This interactive workbook demonstrates example of Elasticsearch's [MultiQuery Retriever](https://api.python.langchain.com/en/latest/retrievers/langchain.retrievers.multi_query.MultiQueryRetriever.html) to generate similar queries for a given user input and apply all queries to retrieve a larger set of relevant documents from a vectorstore.

Before we begin, we first split the fictional workplace documents into passages with `langchain` and uses OpenAI to transform these passages into embeddings and then store these into Elasticsearch.

We will then ask a question, generate similar questions using langchain and OpenAI, retrieve relevant passages from the vector store, and use langchain and OpenAI again to provide a summary for the questions.

## Install packages and import modules

In [1]:
!pip install -qU "langchain>=1.0" "langchain-core>=0.3" "langchain-community>=0.4" "langchain-classic>=0.3" langchain-openai langchain-elasticsearch tiktoken jq lark elasticsearch

In [2]:
import os
from getpass import getpass
from langchain_openai.embeddings import OpenAIEmbeddings
#from langchain_elasticsearch import ElasticsearchStore
from langchain_openai import ChatOpenAI
from langchain_classic.retrievers.multi_query import MultiQueryRetriever  # ← CORRECT import for 1.0+


from langchain_community.vectorstores.elasticsearch import ElasticsearchStore
#from langchain_openai import OpenAIEmbeddings

# os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")

/opt/homebrew/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Connect to Elasticsearch

ℹ️ We're using an Elastic Cloud deployment of Elasticsearch for this notebook. If you don't have an Elastic Cloud deployment, sign up [here](https://cloud.elastic.co/registration?utm_source=github&utm_content=elasticsearch-labs-notebook) for a free trial.

We'll use the **Cloud ID** to identify our deployment, because we are using Elastic Cloud deployment. To find the Cloud ID for your deployment, go to https://cloud.elastic.co/deployments and select your deployment.

We will use [ElasticsearchStore](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html) to connect to our elastic cloud deployment, This would help create and index data easily.  We would also send list of documents that we created in the previous step

In [9]:
# https://www.elastic.co/search-labs/tutorials/install-elasticsearch/elastic-cloud#finding-your-cloud-id
ELASTIC_CLOUD_ID = getpass("Elastic Cloud ID: ")

# https://www.elastic.co/search-labs/tutorials/install-elasticsearch/elastic-cloud#creating-an-api-key
ELASTIC_API_KEY = getpass("Elastic Api Key: ")

# https://platform.openai.com/api-keys
OPENAI_API_KEY = getpass("OpenAI API key: ")


# Create OpenAI embedding model
embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

# Set a meaningful index name
INDEX_NAME = "multi-query-retriever" #give it a meaningful name,

# Create Elasticsearch vector store
vectorstore = ElasticsearchStore(
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
    index_name=INDEX_NAME,
    embedding=embeddings  # ✅ Add this line
)


## Indexing Data into Elasticsearch
Let's download the sample dataset and deserialize the document.

In [10]:
from urllib.request import urlopen
import json

url = "https://raw.githubusercontent.com/elastic/elasticsearch-labs/main/example-apps/chatbot-rag-app/data/data.json"

response = urlopen(url)
data = json.load(response)

with open("temp.json", "w") as json_file:
    json.dump(data, json_file)

### Split Documents into Passages

We’ll chunk documents into passages in order to improve the retrieval specificity and to ensure that we can provide multiple passages within the context window of the final question answering prompt.

Here we are chunking documents into 800 token passages with an overlap of 400 tokens.

Here we are using a simple splitter but Langchain offers more advanced splitters to reduce the chance of context being lost.

In [11]:
from langchain_community.document_loaders import JSONLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import json  # Optional: for metadata extraction
from datetime import datetime # Import datetime here


def metadata_func(record: dict, metadata: dict) -> dict:
    #Populate the metadata dictionary with keys name, summary, url, category, and updated_at.
    None
    return metadata



loader = JSONLoader(
    file_path="temp.json",
    jq_schema=".[]",  # Extracts array of records
    content_key="content",
    metadata_func=metadata_func,
)

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=1000,       # e.g., ~750 words
    chunk_overlap=200,     # Overlap for context preservation
)

docs = loader.load_and_split(text_splitter=text_splitter)


### Bulk Import Passages

Now that we have split each document into the chunk size of 800, we will now index data to elasticsearch using [ElasticsearchStore.from_documents](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html#langchain.vectorstores.elasticsearch.ElasticsearchStore.from_documents).

We will use Cloud ID, Password and Index name values set in the `Create cloud deployment` step.

In [13]:
from datetime import datetime
from langchain_elasticsearch import ElasticsearchStore
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

# Clean docs metadata
clean_docs = []
for doc in docs:
    metadata = doc.metadata.copy()

    if metadata.get("updated_at") in ["", None, "null"]:
        metadata["updated_at"] = datetime.now().isoformat()

    # Use model_copy() instead of copy() to avoid Pydantic deprecation warning
    clean_docs.append(doc.model_copy(update={"metadata": metadata}))

# Create embeddings ONCE
embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

# Create vectorstore ONCE using cleaned docs
vectorstore = ElasticsearchStore.from_documents(
    clean_docs,
    embeddings,
    index_name=INDEX_NAME,
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
)

# Create LLM
llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0,
    openai_api_key=OPENAI_API_KEY # Explicitly pass the API key
)

# Create MultiQueryRetriever
retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={"k": 4}),
    llm=llm
)


# Question Answering with MultiQuery Retriever

Now that we have the passages stored in Elasticsearch, we can now ask a question to get the relevant passages.

In [14]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import chain as lc_chain
import logging

# Enable detailed logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Multi-query generator with 3 variants
MULTI_QUERY_PROMPT = ChatPromptTemplate.from_template("""
Generate 3 diverse versions of this question for better retrieval. Vary phrasing, keywords, and perspectives:

{question}

Queries (one per line):
""")

LLM_DOCUMENT_PROMPT = PromptTemplate.from_template("""
📄 [{source}]
{page_content}
---
""")

# Define the context prompt for the LLM
LLM_CONTEXT_PROMPT = ChatPromptTemplate.from_template("""
Answer the question based only on the following context:
{context}

Question: {question}
""")

def safe_combine_docs(docs):
    """Production-ready doc formatting with fallbacks"""
    doc_strings = []
    for i, doc in enumerate(docs):
        try:
            doc_dict = doc.model_dump()
            source = doc.metadata.get("name") or doc.metadata.get("source", f"Doc-{i}")
            doc_dict["source"] = source
            formatted = LLM_DOCUMENT_PROMPT.format(**doc_dict)
        except Exception as e:
            logger.warning(f"Doc format error: {e}")
            formatted = f"[Doc-{i}] {doc.page_content[:500]}..."
        doc_strings.append(formatted)
    return "\n\n".join(doc_strings)

# Self-healing chain: retry bad retrievals
def self_healing_retriever(query, max_tries=2):
    """Retry with rewritten query if empty results"""
    for attempt in range(max_tries):
        docs = retriever.invoke(query)
        if docs:
            return docs
        logger.info(f"Empty results (attempt {attempt+1}), rewriting...")
        query = llm.invoke(f"Rewrite for better retrieval: {query}").content
    return retriever.invoke(query)  # Fallback

_context = RunnableParallel(
    context=(RunnablePassthrough() | self_healing_retriever | safe_combine_docs),
    question=RunnablePassthrough(),
)

rag_chain = _context | LLM_CONTEXT_PROMPT | llm | StrOutputParser()

# Test with auto-multi-query
def multi_query_rag(question):
    """Generate + retrieve + answer"""

    query_chain = MULTI_QUERY_PROMPT | llm | StrOutputParser()

    generated_queries = query_chain.invoke({"question": question})

    print("\nGenerated Queries:")
    print("------------------")
    print(generated_queries)
    print("------------------\n")

    queries = [q.strip() for q in generated_queries.split("\n") if q.strip()]

    all_docs = []

    for q in queries:
        docs = self_healing_retriever(q)
        all_docs.extend(docs[:3])

    return rag_chain.invoke({
        "question": question,
        "context": safe_combine_docs(all_docs)
    })

print("---- Answer ----")

print(multi_query_rag("what is the nasa sales team?"))

---- Answer ----


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"



Generated Queries:
------------------
1. Can you provide information about the sales team at NASA?
2. What is the role of the sales team within NASA?
3. How does NASA's sales team operate and contribute to the organization's goals?
------------------



INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:langchain_classic.retrievers.multi_query:Generated queries: ['1. What details can you share about the sales department within NASA?', '2. Could you give me insights into the sales team operating within NASA?', "3. I'm interested in learning more about the sales team specifically within NASA. Can you provide relevant information?"]
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:elastic_transport.transport:POST https://8a87ec9ac6b64fe4928f9a51c6e640ae.us-central1.gcp.cloud.es.io:443/multi-query-retriever/_search?_source_includes=metadata,text [status:200 duration:0.209s]
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:elastic_transport.transport:POST https://8a87ec9ac6b64fe4928f9a51c6e640ae.us-central1.gcp.cloud.es.io:443/multi-query-retriever/_search?_source_includes=metadata,text [status:200 duration:0.173s]

The NASA sales team consists of two Area Vice-Presidents: Laura Martinez, who is the Area Vice-President of North America, and Gary Johnson, who is the Area Vice-President of South America.


**Generate at least two new iteratioins of the previous cells - Be creative.** Did you master Multi-
Query Retriever concepts through this lab?

## Iteration 1: Persona-Based Multi-Query RAG

This version asks the LLM to rewrite the same question from different workplace perspectives, retrieves documents for each perspective, deduplicates the results, and then answers with a short comparison of what each perspective surfaced.

In [15]:
PERSONA_QUERY_PROMPT = ChatPromptTemplate.from_template("""
Rewrite the question as 4 focused search queries, each from a different perspective:
- sales operations
- customer support
- product leadership
- compliance or policy review

Original question: {question}

Return only the queries, one per line.
""")

PERSONA_ANSWER_PROMPT = ChatPromptTemplate.from_template("""
Use only the context below to answer the question.

First give a concise direct answer.
Then add a short "perspective notes" section explaining what different angles revealed.

Context:
{context}

Question: {question}
""")

def unique_docs_by_content(docs):
    seen = set()
    unique = []
    for doc in docs:
        key = doc.page_content[:500]
        if key not in seen:
            seen.add(key)
            unique.append(doc)
    return unique

def persona_multi_query_rag(question, docs_per_query=3):
    query_chain = PERSONA_QUERY_PROMPT | llm | StrOutputParser()
    generated_queries = query_chain.invoke({"question": question})
    queries = [q.strip(" -0123456789.") for q in generated_queries.split("\n") if q.strip()]

    print("Generated persona queries:")
    for query in queries:
        print(f"- {query}")

    gathered_docs = []
    base_retriever = vectorstore.as_retriever(search_kwargs={"k": docs_per_query})

    for query in queries:
        gathered_docs.extend(base_retriever.invoke(query))

    context = safe_combine_docs(unique_docs_by_content(gathered_docs))
    answer_chain = PERSONA_ANSWER_PROMPT | llm | StrOutputParser()
    return answer_chain.invoke({"question": question, "context": context})

print(persona_multi_query_rag("what is the nasa sales team?"))

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Generated persona queries:
- What is the structure of the NASA sales operations team?
- How does the NASA customer support team assist users?
- What role does product leadership play in NASA's operations?
- How does NASA ensure compliance with policies and regulations?


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:elastic_transport.transport:POST https://8a87ec9ac6b64fe4928f9a51c6e640ae.us-central1.gcp.cloud.es.io:443/multi-query-retriever/_search?_source_includes=metadata,text [status:200 duration:0.789s]
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:elastic_transport.transport:POST https://8a87ec9ac6b64fe4928f9a51c6e640ae.us-central1.gcp.cloud.es.io:443/multi-query-retriever/_search?_source_includes=metadata,text [status:200 duration:0.156s]
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:elastic_transport.transport:POST https://8a87ec9ac6b64fe4928f9a51c6e640ae.us-central1.gcp.cloud.es.io:443/multi-query-retriever/_search?_source_includes=metadata,text [status:200 duration:0.166s]
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:elastic_transport.transport:POST https://8a87ec9ac6b

The NASA sales team consists of two Area Vice-Presidents: Laura Martinez for North America and Gary Johnson for South America.

Perspective notes:
The NASA sales team is a combination of the North America and South America regions, led by two Area Vice-Presidents. This structure allows for focused leadership and management of sales activities in both regions under the NASA umbrella. The team is responsible for identifying and pursuing new business opportunities, nurturing client relationships, and ensuring customer satisfaction across North and South America.


## Iteration 2: Skeptical RAG With Evidence Audit

This version answers normally, then asks the model to audit its own answer against the retrieved context. It is useful when you want the answer plus a quick check on whether the retrieved evidence really supports it.

In [16]:
SKEPTICAL_QUERY_PROMPT = ChatPromptTemplate.from_template("""
Create 5 search queries for this question:
1. one broad query
2. one query for exact names or teams
3. one query for processes or responsibilities
4. one query for risks, limits, or exceptions
5. one query using likely synonyms

Question: {question}

Return only the queries, one per line.
""")

SKEPTICAL_ANSWER_PROMPT = ChatPromptTemplate.from_template("""
Answer the question using only this context.
If the context is incomplete, say what is missing.

Context:
{context}

Question: {question}
""")

EVIDENCE_AUDIT_PROMPT = ChatPromptTemplate.from_template("""
Audit the answer against the context.

Return:
- verdict: Supported, Partially supported, or Not supported
- strongest evidence: 2 short bullets from the context
- missing evidence: anything the answer claims that the context does not prove

Context:
{context}

Answer:
{answer}
""")

def skeptical_multi_query_rag(question, docs_per_query=2):
    query_chain = SKEPTICAL_QUERY_PROMPT | llm | StrOutputParser()
    generated_queries = query_chain.invoke({"question": question})
    queries = [q.strip(" -0123456789.") for q in generated_queries.split("\n") if q.strip()]

    print("Generated skeptical queries:")
    for query in queries:
        print(f"- {query}")

    gathered_docs = []
    for query in queries:
        gathered_docs.extend(retriever.invoke(query)[:docs_per_query])

    context = safe_combine_docs(unique_docs_by_content(gathered_docs))

    answer_chain = SKEPTICAL_ANSWER_PROMPT | llm | StrOutputParser()
    answer = answer_chain.invoke({"question": question, "context": context})

    audit_chain = EVIDENCE_AUDIT_PROMPT | llm | StrOutputParser()
    audit = audit_chain.invoke({"context": context, "answer": answer})

    return f"ANSWER\n------\n{answer}\n\nEVIDENCE AUDIT\n--------------\n{audit}"

print(skeptical_multi_query_rag("what is the nasa sales team?"))

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Generated skeptical queries:
- NASA sales team information
- NASA sales team members
- NASA sales team roles and responsibilities
- NASA sales team risks and limitations
- NASA sales department personnel


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:langchain_classic.retrievers.multi_query:Generated queries: ['1. Where can I find details about the sales team at NASA?', '2. How can I access information on the sales representatives working at NASA?', '3. What resources are available for learning about the sales department within NASA?']
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:elastic_transport.transport:POST https://8a87ec9ac6b64fe4928f9a51c6e640ae.us-central1.gcp.cloud.es.io:443/multi-query-retriever/_search?_source_includes=metadata,text [status:200 duration:0.272s]
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:elastic_transport.transport:POST https://8a87ec9ac6b64fe4928f9a51c6e640ae.us-central1.gcp.cloud.es.io:443/multi-query-retriever/_search?_source_includes=metadata,text [status:200 duration:0.158s]
INFO:httpx:HTTP Request: POST https://api

ANSWER
------
The NASA sales team consists of Laura Martinez as the Area Vice-President of North America and Gary Johnson as the Area Vice-President of South America.

EVIDENCE AUDIT
--------------
Verdict: Supported

Strongest evidence:
- "The North America South America region (NASA) has two Area Vice-Presidents: Laura Martinez is the Area Vice-President of North America, and Gary Johnson is the Area Vice-President of South America."
- The context clearly states the specific individuals who are the Area Vice-Presidents for the North America South America region.

Missing evidence: 
The answer does not claim anything beyond the specific individuals who are the Area Vice-Presidents for the NASA region.
